[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IgnatiusEzeani/spatial-humanities-2026/blob/sh2026-workshop/workshop/01_manual_annotation.ipynb)

# AI and NLP for Spatial Humanities
## 01 - Manual annotation: humans define the task before models are scored

**Duration:** 35-45 minutes

This notebook begins without an NLP model. We first make the human decisions visible: what counts as spatial evidence, where its boundaries lie, which relations are supported, and what requires inference.

**Learning outcomes:** distinguish location/locale/sense of place; create exact-offset annotations; compare exact and overlap matching; identify selection, boundary, ontology and inference disagreements; save a human annotation for later model comparison.

> A human reference annotation is a documented scholarly decision, not interpretation-free truth.

## 1. Setup

This notebook runs after Notebook 00 or independently. No GPU or API key is required. `FAST_MODE=True` remains the workshop default.

In [1]:
!wget -q https://raw.githubusercontent.com/IgnatiusEzeani/spatial-humanities-2026/sh2026-workshop/workshop/sh2026_setup.py

import sh2026_setup as sh
ctx = sh.setup()

# Bound from the shared context: the cells below were written against these.
repo_dir = ctx.repo
data_dir = ctx.data

import json
FAST_MODE = True
print("Repository:", repo_dir)


Reusing existing checkout at /home/ezeani/workspace/spatial-humanities-2026
Dependencies already installed in this runtime.

Ready in 0s.
  repo    : /home/ezeani/workspace/spatial-humanities-2026
  commit  : 798f2be
  data    : /home/ezeani/workspace/spatial-humanities-2026/workshop/data
  outputs : /home/ezeani/workspace/spatial-humanities-2026/sh2026_outputs
  route   : CPU only, no API key needed

If this cell failed, put your hand up. Do not re-run it more than once.
Repository: /home/ezeani/workspace/spatial-humanities-2026


In [2]:
import pandas as pd
from IPython.display import display
from spatio_textual.gold import (
    SPAN_LABELS, assert_valid_gold, find_span, load_gold_jsonl,
    score_span_annotations, select_spans,
)
gold_path = repo_dir/"workshop"/"data"/"gold_reference_v0.1.jsonl"
records = load_gold_jsonl(gold_path)
assert_valid_gold(records)
gold = {r["example_id"]: r for r in records}
print(f"Loaded {len(records)} validated reference examples.")

Loaded 5 validated reference examples.


## 2. What counts as spatial information?

We use three connected analytical levels:

| Level | Question | Examples |
|---|---|---|
| **Location** | What named place is mentioned? | Penrith, Eamont |
| **Locale** | What spatial setting/feature is described? | road, river, village, woods |
| **Sense of place** | How is place experienced/evaluated? | picturesque, wild, sensory/affective description |

Spatial narratives also encode distance, direction, time, movement, relations and uncertainty. They cannot be reduced to coordinates.

In [3]:
record = gold["cldw_penrith_pooley_bridge"]
text = record["text"]
print(text)
print("\nReference status:", record["reference_status"])
print("Distribution:", record["source"]["distribution_status"])

From Penrith two roads lead to Pooley Bridge, about six miles distant, which spans the Eamont just at its issue from Ulleswater.

Reference status: adjudicated_reference
Distribution: public_domain_source_verified


## 3. Exercise A - annotate freely

Before reading the label guide, mark anything you regard as spatially meaningful.

Ask yourself:
- Which strings are named places?
- Is **roads** spatial information?
- Is **about six miles distant** spatial information?
- Does **spans the Eamont** encode something that a place-name recognizer would miss?
- What would be lost if we retained only latitude/longitude?

The ontology decides what a system is capable of seeing **before any model is run**.

### Working labels
`TOPONYM`, `GEONOUN`, `SPATIAL_RELATION`, `DISTANCE`, `DIRECTION`, `TIME`, `MOVEMENT_CUE`, `TRANSPORT_CUE`, `SUBJECTIVE_DESCRIPTOR`, `SENSORY_DESCRIPTOR`, `DEICTIC_REFERENCE`.

See `docs/GOLD_ANNOTATION_GUIDE.md` for the full policy.

In [4]:
guide = pd.DataFrame([
("TOPONYM","entity","location"),("GEONOUN","entity","locale"),
("SPATIAL_RELATION","spatial_cue",None),("DISTANCE","spatial_cue",None),
("DIRECTION","spatial_cue",None),("TIME","temporal_cue",None),
("MOVEMENT_CUE","event_cue",None),("TRANSPORT_CUE","journey_cue",None),
("SUBJECTIVE_DESCRIPTOR","sense_of_place","sense_of_place"),
("SENSORY_DESCRIPTOR","sense_of_place","sense_of_place"),
("DEICTIC_REFERENCE","spatial_cue","location")],
columns=["label","layer","conceptual_level"])
display(guide)

,label,layer,conceptual_level
0,TOPONYM,entity,location
1,GEONOUN,entity,locale
2,SPATIAL_RELATION,spatial_cue,NaN
3,DISTANCE,spatial_cue,NaN
4,DIRECTION,spatial_cue,NaN
5,TIME,temporal_cue,NaN
6,MOVEMENT_CUE,event_cue,NaN
7,TRANSPORT_CUE,journey_cue,NaN
8,SUBJECTIVE_DESCRIPTOR,sense_of_place,sense_of_place
9,SENSORY_DESCRIPTOR,sense_of_place,sense_of_place


## 4. Exact offsets are part of the evidence

Offsets are half-open: `text[start_char:end_char]` must reproduce the source string exactly. The helper below refuses to invent an occurrence that is not present.

In [5]:
example = find_span(text,"Penrith","TOPONYM",layer="entity",span_id="p001")
assert text[example["start_char"]:example["end_char"]] == example["text"]
example

{'span_id': 'p001',
 'layer': 'entity',
 'label': 'TOPONYM',
 'text': 'Penrith',
 'start_char': 5,
 'end_char': 12,
 'certainty': 'explicit',
 'attributes': {},
 'notes': None}

## 5. Exercise B - build your annotation

Edit the list. The starter contains only one span so the instructor reference is not handed to you.

In [6]:
participant_spans = [
    find_span(text,"Penrith","TOPONYM",layer="entity",span_id="p001"),
    # Add your own decisions, for example:
    # find_span(text,"roads","GEONOUN",layer="entity",span_id="p002"),
]
for s in participant_spans:
    assert s["label"] in SPAN_LABELS
    assert text[s["start_char"]:s["end_char"]] == s["text"]
display(pd.DataFrame(participant_spans))

,span_id,layer,label,text,start_char,end_char,certainty,attributes,notes
0,p001,entity,TOPONYM,Penrith,5,12,explicit,{},None


## 6. Predict the disagreements before revealing the reference

Discuss:
1. **selection** - did you include `roads`?
2. **boundary** - `six miles`, `about six miles`, or `about six miles distant`?
3. **ontology** - is a road locale, infrastructure, both, or outside the task?
4. **anaphora/inference** - does resolving *which* in `which spans the Eamont` require interpretation?
5. **historical form** - should `Ulleswater` be silently normalized?
6. **representation** - should route structure be a span, a relation, or a journey?

These disagreements are methodological evidence, not mere noise.

In [7]:
SHOW_REFERENCE = True  # set only after discussion
reference_spans = record["spans"]
if SHOW_REFERENCE:
    display(pd.DataFrame(reference_spans)[
        ["span_id","text","label","layer","conceptual_level","start_char","end_char",
         "certainty","attributes","notes"]
    ])

,span_id,text,label,layer,conceptual_level,start_char,end_char,certainty,attributes,notes
0,s001,Penrith,TOPONYM,entity,location,5,12,explicit,{'subtype': 'settlement'},NaN
1,s002,roads,GEONOUN,entity,locale,17,22,explicit,{'subtype': 'route_feature'},NaN
2,s003,Pooley Bridge,TOPONYM,entity,location,31,44,explicit,{'subtype': 'settlement_or_crossing_name'},NaN
3,s004,about six miles distant,DISTANCE,spatial_cue,NaN,46,69,explicit,"{'value': 6, 'unit': 'miles', 'approximate': T...",NaN
4,s005,Eamont,TOPONYM,entity,location,87,93,explicit,{'subtype': 'hydrographic_feature'},NaN
5,s006,Ulleswater,TOPONYM,entity,location,117,127,explicit,"{'subtype': 'hydrographic_feature', 'historica...",Preserve the source spelling; do not silently ...


## 7. Score the spans, then read the disagreements

We report:
- **exact**: same label and exact boundaries;
- **overlap**: same label and overlapping evidence, one-to-one matched by maximum IoU.

Neither score replaces qualitative inspection.

In [8]:
scores=[]
for mode in ("exact","overlap"):
    result=score_span_annotations(participant_spans,reference_spans,match=mode,label_sensitive=True)
    scores.append({k:v for k,v in result.items() if k in {"match","precision","recall","f1","tp","fp","fn"}})
display(pd.DataFrame(scores))

exact=score_span_annotations(participant_spans,reference_spans,match="exact")
print("Unmatched participant rows:")
display(pd.DataFrame([participant_spans[i] for i in exact["unmatched_pred_indices"]]))
print("Unmatched reference rows:")
display(pd.DataFrame([reference_spans[i] for i in exact["unmatched_ref_indices"]]))

,match,tp,fp,fn,precision,recall,f1
0,exact,1,0,5,1.0,0.166667,0.285714
1,overlap,1,0,5,1.0,0.166667,0.285714


Unmatched participant rows:


""


Unmatched reference rows:


,span_id,layer,label,text,start_char,end_char,conceptual_level,certainty,attributes,notes
0,s002,entity,GEONOUN,roads,17,22,locale,explicit,{'subtype': 'route_feature'},NaN
1,s003,entity,TOPONYM,Pooley Bridge,31,44,location,explicit,{'subtype': 'settlement_or_crossing_name'},NaN
2,s004,spatial_cue,DISTANCE,about six miles distant,46,69,NaN,explicit,"{'value': 6, 'unit': 'miles', 'approximate': T...",NaN
3,s005,entity,TOPONYM,Eamont,87,93,location,explicit,{'subtype': 'hydrographic_feature'},NaN
4,s006,entity,TOPONYM,Ulleswater,117,127,location,explicit,"{'subtype': 'hydrographic_feature', 'historica...",Preserve the source spelling; do not silently ...


## 8. A reasonable boundary choice can still get exact F1 = 0

The reference marks `about six miles distant`. A second annotator could reasonably mark only `six miles`. Exact matching measures boundary agreement; overlap matching measures shared evidence.

In [9]:
distance_ref = select_spans(record,["DISTANCE"])
annotator_b = [find_span(text,"six miles","DISTANCE",layer="spatial_cue")]
display(pd.DataFrame([
    {"match":"exact", **{k:v for k,v in score_span_annotations(annotator_b,distance_ref,match="exact").items() if k in {"precision","recall","f1"}}},
    {"match":"overlap", **{k:v for k,v in score_span_annotations(annotator_b,distance_ref,match="overlap").items() if k in {"precision","recall","f1"}}},
]))

,match,precision,recall,f1
0,exact,0.0,0.0,0.0
1,overlap,1.0,1.0,1.0


## 9. Spans are not enough: inspect relations

A relation record separates the wording from the interpreted relation and records whether the interpretation is explicit or contextual.

In [10]:
display(pd.DataFrame(record["relations"])[
    ["relation_id","type","source_span_id","target_span_id",
     "evidence_quote","certainty","attributes","notes"]
])

,relation_id,type,source_span_id,target_span_id,evidence_quote,certainty,attributes,notes
0,r001,CONNECTS,s001,s003,From Penrith two roads lead to Pooley Bridge,explicit,{'mediated_by': 'roads'},NaN
1,r002,APPROX_DISTANCE,s001,s003,about six miles distant,explicit,"{'value': 6, 'unit': 'miles', 'approximate': T...","The relation is explicit, but the exact measur..."
2,r003,SPANS,s003,s005,which spans the Eamont,contextual_inference,{},Resolving 'which' to Pooley Bridge requires an...
3,r004,OUTFLOWS_FROM,s005,s006,Eamont just at its issue from Ulleswater,contextual_inference,{},The wording describes the Eamont at its issue ...


Notice that a model can recognize every toponym and still miss route connectivity, approximate distance, anaphoric `SPANS`, or the relation expressed by `issue from`.

This is why later benchmarks separate:
- entity/span extraction;
- entity resolution;
- relation extraction;
- journey reconstruction.

## 10. Save the participant annotation

We preserve the human decision rather than overwriting it with the instructor reference.

In [11]:
out = repo_dir/"sh2026_outputs"/"human_review"
out.mkdir(parents=True,exist_ok=True)
participant_record = {
    "schema_version":"sh2026-participant-0.1",
    "example_id":record["example_id"],
    "text":text,
    "reference_schema":record["schema_version"],
    "annotation_policy":record["annotation_policy"],
    "spans":participant_spans,
    "relations":[],
    "notes":["Created independently of later computational model comparison."],
}
path=out/f"{record['example_id']}_participant.json"
path.write_text(json.dumps(participant_record,indent=2,ensure_ascii=False),encoding="utf-8")
print("Saved:",path)

Saved: /home/ezeani/workspace/spatial-humanities-2026/sh2026_outputs/human_review/cldw_penrith_pooley_bridge_participant.json


## 11. Extension - spatial meaning without conventional toponyms

The synthetic passage below is deliberately rich in locale, relative position and vague spatial language. It should not be forced into a set of coordinates.

In [12]:
rel = gold["synthetic_relational_space"]
print(rel["text"])
print("\nQuestions: Which items are locale? Is `nearest` a distance? Is `to our left` a direction without a compass frame? Can village → woods be a journey? Is 'hide' an explicit reason or an inference?")
display(pd.DataFrame(rel["spans"])[["text","label","layer","certainty","attributes"]])
display(pd.DataFrame(rel["relations"])[["type","evidence_quote","certainty","attributes","notes"]])
display(pd.DataFrame(rel["journeys"])[["start_location","end_location","date","journey_reason",
                                      "explicit_or_inferred","requires_review","review_notes"]])

We left the village before dawn and hid in the woods beyond the river. The nearest road was somewhere to our left, but we avoided it.

Questions: Which items are locale? Is `nearest` a distance? Is `to our left` a direction without a compass frame? Can village → woods be a journey? Is 'hide' an explicit reason or an inference?


,text,label,layer,certainty,attributes
0,left,MOVEMENT_CUE,event_cue,explicit,{}
1,village,GEONOUN,entity,explicit,{}
2,before dawn,TIME,temporal_cue,explicit,{'relative': True}
3,hid,MOVEMENT_CUE,event_cue,explicit,{}
4,woods,GEONOUN,entity,explicit,{}
5,beyond the river,SPATIAL_RELATION,spatial_cue,explicit,{'relation': 'beyond'}
6,river,GEONOUN,entity,explicit,{}
7,nearest,DISTANCE,spatial_cue,explicit,{'qualitative': True}
8,road,GEONOUN,entity,explicit,{}
9,to our left,DIRECTION,spatial_cue,explicit,{'frame': 'narrator_relative'}


,type,evidence_quote,certainty,attributes,notes
0,BEYOND,woods beyond the river,explicit,{},NaN
1,LEFT_OF,road was somewhere to our left,explicit,{'vagueness': 'somewhere'},"The reference frame is narrator-relative, not ..."


,start_location,end_location,date,journey_reason,explicit_or_inferred,requires_review,review_notes
0,village,woods,before dawn,hide,"{'start_location': 'explicit', 'end_location':...",True,[Treating 'hid' as the journey purpose is a co...


## 12. Data governance and takeaways

The oral-history-style examples in the public tutorial are **synthetic** and labelled as such. Controlled-access Holocaust testimony text is not bundled into this public reference set.

Before any model is evaluated, humans have already decided:
1. what the task is;
2. which concepts matter;
3. where boundaries lie;
4. which relations warrant annotation;
5. how much inference is acceptable;
6. how historical/ambiguous geography is treated;
7. what evidence must be preserved.

So our later comparison will ask not only **which method has the highest F1?**, but also:

> **Which method makes its assumptions, uncertainty and evidential basis easiest for a humanities researcher to inspect and correct?**

Next: **02 - Rules and gazetteers**.